In [26]:
from langchain_ollama import ChatOllama

local_llm = "llama3.2:3b"
llm = ChatOllama(model=local_llm, temperature=0)
llm_json_mode = ChatOllama(model=local_llm, temperature=0, format="json")

In [27]:
import os, getpass


def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set_env("TAVILY_API_KEY")
os.environ["TOKENIZERS_PARALLELISM"] = "true"

In [28]:
from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(max_results=3)

In [29]:
import operator
from typing_extensions import TypedDict
from typing import List, Annotated


class GraphState(TypedDict):
    """dictionary that contains information we want to propagate to, and modify in, each graph node."""

    chat_message: str  # User question
    generation: str  # LLM generation
    max_retries: int  # Max number of retries for answer generation
    answers: int  # Number of answers generated
    loop_step: Annotated[int, operator.add]

In [30]:
router_instructions = """You are an expert at routing a user question to either worker1 or worker2.
The worker1 is an LLM containing information on Customer Service any issues related to that must be directed to it.                    
Use the worker2 for information regarding products and things like that which can be only informational not helpful.
Return JSON with single key, datasource, that is 'worker1' or worker2' depending on the question.
"""

validator_instructions = """You are a validator assessing the relevance of the answer generated by the worker.
If the answer is relevant to the user question, grade it as relevant.
Return JSON with single key, binary_score, that is 'yes' or 'no' score to indicate whether the answer is relevant to the user question.
"""

validator_prompt = """Here is the answer generated by the worker: \n\n {generation} \n\n Here is the user question: \n\n {chat_message}."""

In [10]:
import json
from langchain.schema import Document
from langgraph.graph import END
from langchain_core.messages import HumanMessage, SystemMessage

In [33]:
def route_question(state):
    """Route question to web search or RAG"""

    print("---ROUTE QUESTION---")
    route_question = llm_json_mode.invoke(
        [SystemMessage(content=router_instructions)]
        + [HumanMessage(content=state["chat_message"])]
    )
    source = json.loads(route_question.content)["datasource"]
    if source == "worker1":
        print("---ROUTE QUESTION TO WORKER1---")
        return "worker1"
    elif source == "worker2":
        print("---ROUTE QUESTION TO WORKER2---")
        return "worker2"


def worker1(state):
    generation = llm.invoke([HumanMessage(content=state["chat_message"])])
    return {"generation": generation}


def worker2(state):
    generation = llm.invoke([HumanMessage(content=state["chat_message"])])
    return {"generation": generation}


def validator(state):
    print("---VALIDATOR---")

    chat_message = state["chat_message"]
    generation = state["generation"]
    prompt = validator_prompt.format(chat_message=chat_message, generation=generation)

    response = llm_json_mode.invoke(
        [SystemMessage(content=validator_instructions)] + [HumanMessage(content=prompt)]
    )

    if json.loads(response.content)["binary_score"] == "yes":
        return "yes"
    elif state["loop_step"] <= state["max_retries"]:
        return "retry"
    else:
        return "max_retries"

In [36]:
from langgraph.graph import StateGraph
from IPython.display import Image, display

workflow = StateGraph(GraphState)

workflow.add_node("worker1", worker1)
workflow.add_node("worker2", worker2)
workflow.add_node("validator", validator)

workflow.set_conditional_entry_point(
    route_question, {"worker1": "worker1", "worker2": "worker2"}
)

workflow.add_edge("worker1", "validator")
workflow.add_edge("worker2", "validator")

workflow.add_conditional_edges("worker1", validator, {"yes": END, "max_retries": END})

In [37]:
g = workflow.compile()
display(Image(g.get_graph().draw_mermaid_png()))

ValueError: At 'worker1' node, 'validator' branch found unknown target 'max_retries'

In [23]:
inputs = {
    "chat_message": "I need help in tracking my order, it has been delayed for a week can youu help?"
}
for event in g.stream(inputs, stream_mode="values"):
    print(event)

---ROUTE QUESTION---
---ROUTE QUESTION TO WORKER1---
{'chat_message': 'I need help in tracking my order, it has been delayed for a week can youu help?', 'loop_step': 0}
{'chat_message': 'I need help in tracking my order, it has been delayed for a week can youu help?', 'generation': AIMessage(content="I'd be happy to help you track your order. To do so, I'll need some more information from you. Can you please provide me with the following details:\n\n1. Your order number (if available)\n2. The name of the retailer or shipping company\n3. The date you placed the order\n4. The expected delivery date (as per the original tracking information)\n\nWith this information, I can try to help you track your order and find out what's causing the delay.\n\nAdditionally, you can also check the following options:\n\n1. Contact the retailer's customer service directly: They may be able to provide you with more up-to-date information on the status of your order.\n2. Check the shipping company's website